# Sentinel Vision — Demo Notebook

Automated video surveillance analytics using YOLO11x + BoT-SORT + ReID.

**Phases active:** detection → tracking → zones → gates → dwell → loitering → abandoned objects → calibration → interactions → evidence capture

---
## Setup

Run the cell below to clone the repo, install dependencies, and pull the right branch.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/basil-sami/sentinel-vision"
BRANCH = "phase-3-5-advanced-intelligence"
REPO_DIR = "sentinel-vision"

if not Path(REPO_DIR).exists():
    !git clone --branch {BRANCH} --single-branch {REPO_URL}
else:
    %cd {REPO_DIR}
    !git checkout {BRANCH} && git pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt
%load_ext autoreload
%autoreload 2
print("Ready.")

## Download or Upload Video

Tries to download the [default test video](https://www.youtube.com/watch?v=GJNjaRJWVP8) via yt-dlp.
If that fails (blocked / no yt-dlp), you can upload a local file.

In [ ]:
DEFAULT_VIDEO = "The CCTV People Demo 2.mp4"
DEFAULT_URL = "https://www.youtube.com/watch?v=GJNjaRJWVP8"

if not Path(DEFAULT_VIDEO).exists():
    print("Downloading 1080p test video...")
    ret = !yt-dlp -f 'bestvideo[height<=1080]+bestaudio/best[height<=1080]' \
        --merge-output-format mp4 -o "{DEFAULT_VIDEO}" "{DEFAULT_URL}"

if Path(DEFAULT_VIDEO).exists() and Path(DEFAULT_VIDEO).stat().st_size > 0:
    input_video = DEFAULT_VIDEO
    print(f"Using: {input_video}")
else:
    print("yt-dlp download failed. Please upload a CCTV video file.")
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        input_video = list(uploaded.keys())[0]
        print(f"Using uploaded: {input_video}")
    else:
        raise FileNotFoundError("No video file provided.")

## Run Analysis

Processes the video with detection, tracking, zones, calibration, interactions, and evidence capture.

In [ ]:
import torch
import json
from src.pipeline import analyze_video

zone_config = json.load(open("configs/demo_zones.json"))
calibration_config = json.load(open("configs/demo_calibration.json"))

result = analyze_video(
    video_path=input_video,
    output_dir="outputs",
    model_family="yolo11",      # or "yolo12", "rtdetr"
    model_size="xlarge",
    conf_threshold=0.4,
    device="cuda" if torch.cuda.is_available() else "cpu",
    use_reid=True,
    reid_model="x1_0",          # osnet_x1_0_msmt17 (upgraded from x0_25)
    track_thresh=0.4,
    match_thresh=0.7,
    track_low_thresh=0.1,
    track_buffer=450,
    zone_config=zone_config,
    calibration_config=calibration_config,
    capture_evidence=True,
    filter_stationary_objects=True,
    min_move_distance=20.0,
)
print("\nDone.")

## Summary Report

In [ ]:
from pathlib import Path
summary = Path("outputs/summary.txt")
if summary.exists():
    print(summary.read_text())

## Download Outputs

Download the annotated video, analytics JSON, and summary.

In [ ]:
from google.colab import files
files.download("outputs/output_tracking.mp4")
files.download("outputs/analytics.json")
files.download("outputs/summary.txt")

## Optimization: TensorRT (NVIDIA GPU only)

Exports YOLO to TensorRT engine for 2-5x inference speedup.
Run this once per model, then pass `use_tensorrt=True` to `analyze_video()`.

In [ ]:
# One-time export (run once per model size)
from src.optimization.tensorrt_export import export_to_engine, has_engine

if torch.cuda.is_available() and not has_engine("yolo11", "xlarge"):
    path = export_to_engine(model_family="yolo11", model_size="xlarge", device=0)
    print(f"TensorRT engine: {path}")
else:
    print("Engine exists or CUDA unavailable, skipping export.")

## Multi-Camera Pipeline (production use)

Process multiple video streams in parallel using multiprocessing.
Each camera runs in its own process with independent detector + tracker.

In [ ]:
# Example: process 3 video files in parallel
from src.optimization.multi_stream import MultiCameraPipeline

configs = [
    {"video_path": "cam1.mp4",   "model_size": "nano", "use_tensorrt": True},
    {"video_path": "cam2.mp4",   "model_size": "nano", "use_tensorrt": True},
    {"video_path": "cam3.mp4",   "model_size": "nano", "use_tensorrt": True},
]

# pipeline = MultiCameraPipeline(configs, output_dir="outputs/multi")
# result = pipeline.run()
# print(f"Events across all cameras: {result['event_count']}")